In [1]:
# %%
import sys
import os
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, Dropdown, VBox, HBox

# Add src to path
#sys.path.append(os.path.abspath('../src'))

from auralab.sethares import sethares

In [ ]:
# %%
def plot_sethares_1d_plotly(base_freq=500, n_partials=6, model='min', use_loudness=True):
    freq = base_freq * np.arange(1, n_partials + 1)
    amp = 0.88 ** np.arange(n_partials)
    
    r_low = 1.0
    r_high = 2.3
    n_points = 500
    alphas = np.linspace(r_low, r_high, n_points)
    diss = np.zeros(n_points)
    
    a = np.concatenate((amp, amp))
    
    for i, alpha in enumerate(alphas):
        f = np.concatenate((freq, alpha * freq))
        diss[i] = sethares(f, a, model=model, use_loudness=use_loudness)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=alphas, y=diss, mode='lines', name='Dissonance'))
    
    # Add vertical lines for common intervals
    intervals = [(1, 1), (6, 5), (5, 4), (4, 3), (3, 2), (5, 3), (2, 1)]
    for n, d in intervals:
        ratio = n/d
        if r_low <= ratio <= r_high:
            fig.add_vline(x=ratio, line_width=1, line_dash="dash", line_color="silver")
            fig.add_annotation(x=ratio, y=max(diss), text=f'{n}/{d}', showarrow=False, yshift=10)

    fig.update_layout(
        title='Sethares Dissonance Curve (1D)',
        xaxis_title='Frequency Ratio',
        yaxis_title='Sensory Dissonance',
        template='plotly_white',
        height=500
    )
    fig.show()

interact(plot_sethares_1d_plotly, 
         base_freq=FloatSlider(min=100, max=1000, step=10, value=500, continuous_update=False),
         n_partials=IntSlider(min=1, max=10, value=6, continuous_update=False),
         model=['min', 'product'],
         use_loudness=Checkbox(value=True, description='Use Loudness Conversion'))

interactive(children=(FloatSlider(value=500.0, continuous_update=False, description='base_freq', max=1000.0, m…

<function __main__.plot_sethares_1d_plotly(base_freq=500, n_partials=6, model='min', use_loudness=True)>

In [ ]:
# %%
def plot_dissonance_surface_plotly(base_freq=500, n_partials=6, range_max=2.0, resolution=30, use_loudness=True, plot_type='3D Surface'):
    freq = base_freq * np.arange(1, n_partials + 1)
    amp = 0.88 ** np.arange(n_partials)
    
    x = np.linspace(1.0, range_max, resolution)
    y = np.linspace(1.0, range_max, resolution)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    
    a_total = np.concatenate((amp, amp, amp))
    
    for i in range(resolution):
        for j in range(resolution):
            alpha = X[i, j]
            beta = Y[i, j]
            f_total = np.concatenate((freq, alpha * freq, beta * freq))
            Z[i, j] = sethares(f_total, a_total, use_loudness=use_loudness)

    if plot_type == '3D Surface':
        fig = go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale='Viridis')])
        fig.update_layout(
            title='Dissonance Surface (Triad) - 3D',
            scene = dict(
                xaxis_title='Interval 1 Ratio',
                yaxis_title='Interval 2 Ratio',
                zaxis_title='Dissonance'),
            height=700,
            margin=dict(l=65, r=50, b=65, t=90)
        )
    else:
        fig = go.Figure(data=[go.Contour(z=Z, x=x, y=y, colorscale='Viridis')])
        fig.update_layout(
            title='Dissonance Surface (Triad) - Contour',
            xaxis_title='Interval 1 Ratio',
            yaxis_title='Interval 2 Ratio',
            height=700
        )
        # Add diagonal line
        fig.add_shape(type="line",
            x0=1, y0=1, x1=range_max, y1=range_max,
            line=dict(color="white", width=2, dash="dash")
        )

    fig.show()

interact(plot_dissonance_surface_plotly,
         base_freq=FloatSlider(min=100, max=1000, step=10, value=500, continuous_update=False),
         n_partials=IntSlider(min=1, max=10, value=6, continuous_update=False),
         range_max=FloatSlider(min=1.5, max=2.5, step=0.1, value=2.0, continuous_update=False),
         resolution=IntSlider(min=10, max=100, step=10, value=30, continuous_update=False),
         use_loudness=Checkbox(value=True, description='Use Loudness Conversion'),
         plot_type=Dropdown(options=['3D Surface', 'Contour'], value='3D Surface', description='Plot Type'))

interactive(children=(FloatSlider(value=500.0, continuous_update=False, description='base_freq', max=1000.0, m…

<function __main__.plot_dissonance_surface_plotly(base_freq=500, n_partials=6, range_max=2.0, resolution=30, use_loudness=True, plot_type='3D Surface')>